In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load the dataset
# --------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "public"
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMAGES = DATA_DIR / "train"
TEST_IMAGES = DATA_DIR / "test"

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Missing CSV: {TRAIN_CSV}")

if not TRAIN_IMAGES.is_dir():
    raise FileNotFoundError(f"Missing training images: {TRAIN_IMAGES}")

df = pd.read_csv(TRAIN_CSV)

print("Dataset shape:", df.shape)
print("Unique image IDs:", df["image_id"].nunique())
print("Defect classes:", sorted(df["defect_type"].dropna().unique()))


# --------------------------------------------------
# 2. Select the first 1,200 unique image IDs
# --------------------------------------------------

image_ids = sorted(df["image_id"].unique())[:1200]

subset = df[df["image_id"].isin(image_ids)].copy()

print("\nSelected images:", len(image_ids))
print("Selected annotation rows:", len(subset))


# --------------------------------------------------
# 3. Identify positive defect masks
# --------------------------------------------------

subset["has_defect"] = (
    subset["mask_rle"].notna()
    & (subset["mask_rle"].astype(str).str.strip() != "")
)


# --------------------------------------------------
# 4. Create image-level combination strings
# --------------------------------------------------

def get_combination(group):
    positive_classes = sorted(
        group.loc[group["has_defect"], "defect_type"].tolist()
    )

    if len(positive_classes) == 0:
        return "clean"

    return "+".join(positive_classes)


combinations = (
    subset.groupby("image_id")
    .apply(get_combination)
    .rename("combination")
    .reset_index()
)


# --------------------------------------------------
# 5. Calculate Question 1 answers
# --------------------------------------------------

# (a) Completely clean images
clean_count = (
    combinations["combination"] == "clean"
).sum()


# (b) Positive images for each defect class
class_counts = (
    subset[subset["has_defect"]]
    .groupby("defect_type")["image_id"]
    .nunique()
)

defect_a = class_counts.get("defect_a", 0)
defect_b = class_counts.get("defect_b", 0)
defect_c = class_counts.get("defect_c", 0)
defect_d = class_counts.get("defect_d", 0)


# (c) Number of unique combination strings
unique_combinations = combinations["combination"].nunique()


# (d) Largest and smallest combination group sizes
group_sizes = combinations["combination"].value_counts()

largest_group = group_sizes.max()
smallest_group = group_sizes.min()


# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("\n========== QUESTION 1 RESULTS ==========")

print("(a) Completely clean images:", clean_count)

print("\n(b) Positive images per defect class:")
print("defect_a:", defect_a)
print("defect_b:", defect_b)
print("defect_c:", defect_c)
print("defect_d:", defect_d)

print("\n(c) Unique combination strings:", unique_combinations)

print("\n(d) Combination group sizes:")
print("Largest:", largest_group)
print("Smallest:", smallest_group)

print("\nAll combination groups:")
display(group_sizes.to_frame("image_count"))


# --------------------------------------------------
# 7. Final answer in submission format
# --------------------------------------------------

print("\nFINAL ANSWER:")
print(
    f"{clean_count}; "
    f"{defect_a}, {defect_b}, {defect_c}, {defect_d}; "
    f"{unique_combinations}; "
    f"{largest_group}, {smallest_group}"
)

Dataset shape: (45572, 5)
Unique image IDs: 11393
Defect classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']

Selected images: 1200
Selected annotation rows: 4800

========== QUESTION 1 RESULTS ==========
(a) Completely clean images: 578

(b) Positive images per defect class:
defect_a: 107
defect_b: 35
defect_c: 438
defect_d: 181

(c) Unique combination strings: 15

(d) Combination group sizes:
Largest: 578
Smallest: 1

All combination groups:


,image_count
combination,
clean,578
defect_c,334
defect_d,81
defect_c+defect_d,71
defect_a,59
defect_b,22
defect_a+defect_c,21
defect_a+defect_d,14
defect_a+defect_c+defect_d,7



FINAL ANSWER:
578; 107, 35, 438, 181; 15; 578, 1


In [3]:
# Select the same 1,200-image subset
subset = df[df["image_id"].isin(image_ids)].copy()

# Select defect_b annotations
defect_b_df = subset[
    subset["defect_type"] == "defect_b"
].copy()

HEIGHT = 128
WIDTH = 800
TOTAL_PIXELS = HEIGHT * WIDTH

# Decode RLE
def decode_rle(rle):
    mask = np.zeros(TOTAL_PIXELS, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((HEIGHT, WIDTH), order="F")

    values = np.asarray(str(rle).split(), dtype=int)

    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        end = min(start + length, TOTAL_PIXELS)

        if start < TOTAL_PIXELS and end > start:
            mask[start:end] = 1

    return mask.reshape((HEIGHT, WIDTH), order="F")


# Calculate defect_b area for each image
areas = []

for _, row in defect_b_df.iterrows():
    mask = decode_rle(row["mask_rle"])
    areas.append((row["image_id"], int(mask.sum())))

areas_df = pd.DataFrame(
    areas, columns=["image_id", "defect_b_area"]
)

# Include images with zero defect_b area
areas_df = (
    pd.DataFrame({"image_id": image_ids})
    .merge(areas_df, on="image_id", how="left")
)

areas_df["defect_b_area"] = areas_df["defect_b_area"].fillna(0)

# Calculate metrics
total_positive = int(areas_df["defect_b_area"].sum())
total_pixels = 1200 * TOTAL_PIXELS
total_negative = total_pixels - total_positive

positive_images = areas_df[
    areas_df["defect_b_area"] > 0
]

percentage = total_positive / total_pixels * 100
mean_area = positive_images["defect_b_area"].mean()
pos_weight = total_negative / total_positive

print("Positive pixels:", total_positive)
print("Positive images:", len(positive_images))
print("Positive pixel percentage:", f"{percentage:.4f}%")
print("Mean defect area:", f"{mean_area:.1f}")
print("Negative/positive ratio:", round(pos_weight))

print("\nFINAL ANSWER:")
print(f"{percentage:.4f}%; {mean_area:.1f}; {round(pos_weight)}")

Positive pixels: 19654
Positive images: 35
Positive pixel percentage: 0.0160%
Mean defect area: 561.5
Negative/positive ratio: 6251

FINAL ANSWER:
0.0160%; 561.5; 6251


In [4]:
# Get combination group sizes from Question 1
group_sizes = combinations["combination"].value_counts()

n_clean = int(group_sizes["clean"])
total_images = int(group_sizes.sum())

# Calculate the weight for each combination group
group_weights = pd.DataFrame({
    "group": group_sizes.index,
    "group_size": group_sizes.values
})

group_weights["weight"] = np.minimum(
    np.sqrt(n_clean / group_weights["group_size"]),
    5
)

# Total sampling probability mass for each group
group_weights["sampling_mass"] = (
    group_weights["group_size"] * group_weights["weight"]
)

total_mass = group_weights["sampling_mass"].sum()

# (a) Expected clean fraction
clean_mass = group_weights.loc[
    group_weights["group"] == "clean", "sampling_mass"
].sum()

a_sampler = clean_mass / total_mass * 100
a_natural = n_clean / total_images * 100

# (b) Expected defect_b-positive draws
b_images = combinations[
    combinations["combination"].str.contains(
        "defect_b", regex=False
    )
]["image_id"].nunique()

b_sampler = 1200 * (
    group_weights.loc[
        group_weights["group"].str.contains("defect_b", regex=False),
        "sampling_mass"
    ].sum() / total_mass
)

b_natural = 1200 * b_images / total_images

# (c) Enrichment factor
enrichment = b_sampler / b_natural

# (d) Number of groups whose weight reaches the cap of 5
capped_groups = (
    group_weights["weight"] >= 5
).sum()

# Display results
print("Total images:", total_images)
print("Clean images:", n_clean)
print("Defect_b-positive images:", b_images)

print("\n(a) Clean fraction:")
print("Sampler:", f"{a_sampler:.2f}%")
print("Natural:", f"{a_natural:.2f}%")

print("\n(b) Expected defect_b-positive draws:")
print("Sampler:", f"{b_sampler:.2f}")
print("Natural:", f"{b_natural:.2f}")

print("\n(c) Enrichment factor:", f"{enrichment:.2f}x")

print("\n(d) Groups reaching weight cap:", capped_groups)

print("\nGroup weights:")
display(group_weights)

print("\nFINAL ANSWER:")
print(
    f"{a_sampler:.2f}%; {a_natural:.2f}%; "
    f"{b_sampler:.2f}; {b_natural:.2f}; "
    f"{enrichment:.2f}x; {capped_groups}"
)

Total images: 1200
Clean images: 578
Defect_b-positive images: 35

(a) Clean fraction:
Sampler: 28.81%
Natural: 48.17%

(b) Expected defect_b-positive draws:
Sampler: 104.69
Natural: 35.00

(c) Enrichment factor: 2.99x

(d) Groups reaching weight cap: 10

Group weights:


,group,group_size,weight,sampling_mass
0,clean,578,1.000000,578.000000
1,defect_c,334,1.315499,439.376831
2,defect_d,81,2.671292,216.374675
3,defect_c+defect_d,71,2.853217,202.578380
4,defect_a,59,3.129954,184.667268
5,defect_b,22,5.000000,110.000000
6,defect_a+defect_c,21,5.000000,105.000000
7,defect_a+defect_d,14,5.000000,70.000000
8,defect_a+defect_c+defect_d,7,5.000000,35.000000
9,defect_a+defect_b+defect_d,3,5.000000,15.000000



FINAL ANSWER:
28.81%; 48.17%; 104.69; 35.00; 2.99x; 10


In [5]:
import math

base_lr = 3e-4

def get_lr(step):
    if step < 60:
        multiplier = step / 60
    else:
        multiplier = 0.5 * (
            1 + math.cos(math.pi * (step - 60) / 420)
        )

    return base_lr * multiplier


# (a) Learning rates at the requested optimizer steps
lr_15 = get_lr(15)
lr_270 = get_lr(270)
lr_400 = get_lr(400)

# (b) Incorrect scheduler stepping: once per epoch
lr_bug = get_lr(8)

print("(a) Learning rates:")
print("Step 15:", f"{lr_15:.3e}")
print("Step 270:", f"{lr_270:.3e}")
print("Step 400:", f"{lr_400:.3e}")

print("\n(b) LR after 8 epochs with incorrect stepping:")
print(f"{lr_bug:.3e}")

(a) Learning rates:
Step 15: 7.500e-05
Step 270: 1.500e-04
Step 400: 2.606e-05

(b) LR after 8 epochs with incorrect stepping:
4.000e-05


In [6]:
val_scores = [
    0.512, 0.601, 0.648, 0.661,
    0.664, 0.668, 0.671, 0.670,
    0.672, 0.669, 0.673, 0.665
]

patience = 3
min_delta = 0.002

best_score = float("-inf")
best_epoch = 0
counter = 0
stop_epoch = None

for epoch, score in enumerate(val_scores, start=1):

    if score > best_score + min_delta:
        best_score = score
        best_epoch = epoch
        counter = 0
    else:
        counter += 1

    print(
        f"Epoch {epoch}: score={score:.3f}, "
        f"best={best_score:.3f}, patience={counter}"
    )

    if counter >= patience:
        stop_epoch = epoch
        break

print("\nStopping epoch:", stop_epoch)
print("Restored checkpoint epoch:", best_epoch)
print("Restored score:", best_score)


# Part (b): Repeat with min_delta = 0

best_score_b = float("-inf")
best_epoch_b = 0
counter_b = 0
stop_epoch_b = None

for epoch, score in enumerate(val_scores, start=1):

    if score > best_score_b:
        best_score_b = score
        best_epoch_b = epoch
        counter_b = 0
    else:
        counter_b += 1

    if counter_b >= patience:
        stop_epoch_b = epoch
        break

print("\nWith min_delta = 0:")
print("Stopped within 12 epochs:", "Yes" if stop_epoch_b else "No")
print("Best checkpoint epoch:", best_epoch_b)
print("Best score:", best_score_b)

Epoch 1: score=0.512, best=0.512, patience=0
Epoch 2: score=0.601, best=0.601, patience=0
Epoch 3: score=0.648, best=0.648, patience=0
Epoch 4: score=0.661, best=0.661, patience=0
Epoch 5: score=0.664, best=0.664, patience=0
Epoch 6: score=0.668, best=0.668, patience=0
Epoch 7: score=0.671, best=0.671, patience=0
Epoch 8: score=0.670, best=0.671, patience=1
Epoch 9: score=0.672, best=0.671, patience=2
Epoch 10: score=0.669, best=0.671, patience=3

Stopping epoch: 10
Restored checkpoint epoch: 7
Restored score: 0.671

With min_delta = 0:
Stopped within 12 epochs: No
Best checkpoint epoch: 11
Best score: 0.673


In [7]:
# Model configuration
parameters = 25_000_000
bytes_per_value = 4

# Part A (a): Model memory
model_memory = parameters * bytes_per_value * 4 / 1_000_000

print("(a) Weights + gradients + AdamW states:", model_memory, "MB")


# Part A (b): Memory at batch size 16
batch_size = 16

activation_fp32 = 150
activation_amp = 75

memory_fp32 = model_memory + batch_size * activation_fp32
memory_amp = model_memory + batch_size * activation_amp

print("\n(b) Total memory at batch size 16:")
print("FP32:", memory_fp32, "MB")
print("AMP:", memory_amp, "MB")


# Part A (c): Maximum batch size under 3400 MB
memory_budget = 3400

max_batch_fp32 = int(
    (memory_budget - model_memory) // activation_fp32
)

max_batch_amp = int(
    (memory_budget - model_memory) // activation_amp
)

print("\n(c) Maximum batch sizes:")
print("FP32:", max_batch_fp32)
print("AMP:", max_batch_amp)


# Part B (d): Effective batch size
training_images = 960
micro_batch = 8
accumulation_steps = 4

effective_batch = micro_batch * accumulation_steps

print("\n(d) Effective batch size:", effective_batch)


# Part B (e): Micro-batches and optimizer steps
micro_batches = training_images // micro_batch
optimizer_steps = micro_batches // accumulation_steps

print("\n(e) Micro-batches per epoch:", micro_batches)
print("Optimizer steps per epoch:", optimizer_steps)


# Part B (f): Loss scaling factor
loss_divisor = accumulation_steps

print("\n(f) Divide each micro-batch loss by:", loss_divisor)


# Part B (g): Two-epoch warmup
warmup_epochs = 2
warmup_steps = warmup_epochs * optimizer_steps

print("\n(g) Warmup optimizer steps:", warmup_steps)


# Final answer
print("\nFINAL ANSWER:")
print(
    f"{model_memory:.0f}; "
    f"{memory_fp32:.0f}, {memory_amp:.0f}; "
    f"{max_batch_fp32}, {max_batch_amp}; "
    f"{effective_batch}; "
    f"{micro_batches}, {optimizer_steps}; "
    f"{loss_divisor}; "
    f"{warmup_steps}"
)

(a) Weights + gradients + AdamW states: 400.0 MB

(b) Total memory at batch size 16:
FP32: 2800.0 MB
AMP: 1600.0 MB

(c) Maximum batch sizes:
FP32: 20
AMP: 40

(d) Effective batch size: 32

(e) Micro-batches per epoch: 120
Optimizer steps per epoch: 30

(f) Divide each micro-batch loss by: 4

(g) Warmup optimizer steps: 60

FINAL ANSWER:
400; 2800, 1600; 20, 40; 32; 120, 30; 4; 60


In [9]:
import numpy as np
import pandas as pd

# -----------------------------------------
# Part A: Actual 1,200-image subset
# -----------------------------------------

subset = df[df["image_id"].isin(image_ids)].copy()

# Identify empty and positive masks
subset["has_defect"] = (
    subset["mask_rle"].notna()
    & (subset["mask_rle"].astype(str).str.strip() != "")
)

total_pairs = len(subset)
positive_pairs = subset["has_defect"].sum()
empty_pairs = total_pairs - positive_pairs

# All-empty baseline
naive_dice = empty_pairs / total_pairs

foreground_dice = 0.0
false_positive_score = 1.0
empty_accuracy = 1.0

official_score = (
    0.70 * foreground_dice
    + 0.20 * false_positive_score
    + 0.10 * empty_accuracy
)

print("PART A")
print("Total pairs:", total_pairs)
print("Empty pairs:", empty_pairs)
print("Positive pairs:", positive_pairs)
print("Naive Dice:", f"{naive_dice:.4f}")
print("Official score:", f"{official_score:.4f}")


# -----------------------------------------
# Part B: Toy validation example
# -----------------------------------------

total_pairs_b = 400
positive_pairs_b = 60
empty_pairs_b = 340

foreground_dice_b = 0.50

correct_empty = 306
false_positive_pixels = 1024
total_pixels = 102400

false_positive_score_b = (
    1 - false_positive_pixels / total_pixels
)

empty_accuracy_b = correct_empty / empty_pairs_b

official_score_b = (
    0.70 * foreground_dice_b
    + 0.20 * false_positive_score_b
    + 0.10 * empty_accuracy_b
)

naive_dice_b = (
    positive_pairs_b * foreground_dice_b
    + correct_empty
) / total_pairs_b

baseline_naive_dice_b = empty_pairs_b / total_pairs_b


print("\nPART B")
print("False positive score:", f"{false_positive_score_b:.4f}")
print("Empty accuracy:", f"{empty_accuracy_b:.4f}")
print("Official score:", f"{official_score_b:.4f}")
print("Model naive Dice:", f"{naive_dice_b:.4f}")
print("Baseline naive Dice:", f"{baseline_naive_dice_b:.4f}")


# -----------------------------------------
# Final answer
# -----------------------------------------

print("\nFINAL ANSWER:")
print(
    f"{empty_pairs}; "
    f"{naive_dice:.4f}; "
    f"{official_score:.4f}; "
    f"{official_score_b:.4f}, {naive_dice_b:.4f}; "
    f"naive Dice, {baseline_naive_dice_b:.4f}"
)

PART A
Total pairs: 4800
Empty pairs: 4039
Positive pairs: 761
Naive Dice: 0.8415
Official score: 0.3000

PART B
False positive score: 0.9900
Empty accuracy: 0.9000
Official score: 0.6380
Model naive Dice: 0.8400
Baseline naive Dice: 0.8500

FINAL ANSWER:
4039; 0.8415; 0.3000; 0.6380, 0.8400; naive Dice, 0.8500


In [11]:
# --------------------------------------------------
# 5. Stratified 80/20 split with rare-group handling
# --------------------------------------------------

# Find images belonging to rare_other
rare_ids = m4_combinations.loc[
    m4_combinations["stratify_group"] == "rare_other",
    "image_id"
].values

# Separate rare and regular images
regular_data = m4_combinations[
    m4_combinations["stratify_group"] != "rare_other"
].copy()

regular_ids = regular_data["image_id"].values
regular_labels = regular_data["stratify_group"].values

if len(rare_ids) == 1:

    # Keep the single rare image in training
    regular_train_ids, m4_val_ids = train_test_split(
        regular_ids,
        train_size=959,
        test_size=240,
        random_state=42,
        stratify=regular_labels
    )

    m4_train_ids = np.concatenate([
        regular_train_ids,
        rare_ids
    ])

else:

    # Normal stratified split when rare_other has at least 2 images
    m4_train_ids, m4_val_ids = train_test_split(
        m4_combinations["image_id"].values,
        test_size=0.20,
        random_state=42,
        stratify=m4_combinations["stratify_group"]
    )


# Create training and validation DataFrames
m4_train_df = m4_df[
    m4_df["image_id"].isin(m4_train_ids)
].copy()

m4_val_df = m4_df[
    m4_df["image_id"].isin(m4_val_ids)
].copy()


# Display results
print("Rare images:", len(rare_ids))
print("Training images:", len(m4_train_ids))
print("Validation images:", len(m4_val_ids))

print("Training annotation rows:", len(m4_train_df))
print("Validation annotation rows:", len(m4_val_df))

Rare images: 1
Training images: 960
Validation images: 240
Training annotation rows: 3840
Validation annotation rows: 960


In [ ]:
# ============================================================
# MILESTONE 4 - QUESTION 8
# Six-epoch baseline training and validation
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# -------------------- 1. Reproducibility --------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ["defect_a", "defect_b", "defect_c", "defect_d"]
NUM_CLASSES = 4
HEIGHT, WIDTH = 128, 800

BATCH_SIZE = 16
EPOCHS = 6
BASE_LR = 3e-4
WARMUP_STEPS = 60

print("Device:", device)
print("Training images:", len(m4_train_ids))
print("Validation images:", len(m4_val_ids))


# -------------------- 2. Locate images --------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_IMAGES = PROJECT_ROOT / "data" / "public" / "train"

image_files = {
    p.stem: p for p in TRAIN_IMAGES.iterdir()
    if p.is_file()
}


# -------------------- 3. RLE decoding --------------------

def decode_rle(rle, height=HEIGHT, width=WIDTH):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(str(rle).split(), dtype=np.int64)

    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        if start >= 0 and start < height * width:
            end = min(start + length, height * width)
            mask[start:end] = 1

    return mask.reshape((height, width), order="F")


# -------------------- 4. Dataset --------------------

class M4SteelDataset(Dataset):

    def __init__(self, image_ids, dataframe):
        self.image_ids = list(image_ids)

        self.grouped = {
            image_id: group.set_index("defect_type")
            for image_id, group in dataframe.groupby("image_id")
        }

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):

        image_id = self.image_ids[index]

        image = Image.open(
            image_files[str(image_id)]
        ).convert("L")

        image = image.resize((WIDTH, HEIGHT))
        image = np.asarray(image, dtype=np.float32) / 255.0

        # Same normalization as Milestone 3
        image = (image - 0.449) / 0.226

        image = torch.from_numpy(image).unsqueeze(0)

        masks = np.zeros(
            (NUM_CLASSES, HEIGHT, WIDTH),
            dtype=np.float32
        )

        image_rows = self.grouped[image_id]

        for class_index, class_name in enumerate(CLASS_NAMES):

            if class_name in image_rows.index:
                rle = image_rows.loc[class_name, "mask_rle"]

                masks[class_index] = decode_rle(rle)

        masks = torch.from_numpy(masks)

        return image, masks


train_dataset = M4SteelDataset(m4_train_ids, m4_train_df)
val_dataset = M4SteelDataset(m4_val_ids, m4_val_df)


# -------------------- 5. Bounded square-root sampler --------------------

combination_map = m4_combinations.set_index(
    "image_id"
)["stratify_group"].to_dict()

group_counts = m4_combinations["stratify_group"].value_counts()
clean_count = int(group_counts.get("clean", 1))

sample_weights = []

for image_id in m4_train_ids:

    group = combination_map[image_id]
    group_count = int(group_counts[group])

    weight = min(math.sqrt(clean_count / group_count), 5.0)
    sample_weights.append(weight)

train_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


# -------------------- 6. Compound loss from M3 --------------------

class CompoundLoss(nn.Module):

    def __init__(self, alpha=0.5, beta=0.5, smooth=1.0):
        super().__init__()

        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):

        bce_loss = self.bce(logits, targets)

        probabilities = torch.sigmoid(logits)

        intersection = (probabilities * targets).sum(
            dim=(2, 3)
        )

        denominator = (
            probabilities.sum(dim=(2, 3))
            + targets.sum(dim=(2, 3))
        )

        dice = (
            2 * intersection + self.smooth
        ) / (
            denominator + self.smooth
        )

        dice_loss = 1 - dice.mean()

        return (
            self.alpha * bce_loss
            + self.beta * dice_loss
        )


criterion = CompoundLoss().to(device)


# -------------------- 7. Model: ResNet34 U-Net --------------------

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=1,
    classes=NUM_CLASSES
).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=1e-2
)

# AMP enabled when CUDA is available
amp_enabled = device.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled
)


# -------------------- 8. Warmup + cosine scheduler --------------------

steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch

def get_lr(step):

    if step < WARMUP_STEPS:
        multiplier = (step + 1) / WARMUP_STEPS

    else:
        progress = (
            (step - WARMUP_STEPS)
            / max(1, total_steps - WARMUP_STEPS)
        )

        progress = min(progress, 1.0)

        multiplier = 0.5 * (
            1 + math.cos(math.pi * progress)
        )

    return BASE_LR * multiplier


# -------------------- 9. Official validation metric --------------------

def calculate_official_score(model, loader):

    model.eval()

    intersection = torch.zeros(NUM_CLASSES, device=device)
    pred_sum = torch.zeros(NUM_CLASSES, device=device)
    target_sum = torch.zeros(NUM_CLASSES, device=device)

    positive_pairs = 0
    correct_empty_pairs = 0
    empty_pairs = 0
    false_positive_pixels = 0
    total_pixels = 0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled
            ):

                logits = model(images)

            predictions = (
                torch.sigmoid(logits.float()) >= 0.5
            ).float()

            intersection += (predictions * masks).sum(
                dim=(0, 2, 3)
            )

            pred_sum += predictions.sum(dim=(0, 2, 3))
            target_sum += masks.sum(dim=(0, 2, 3))

            for b in range(masks.size(0)):
                for c in range(NUM_CLASSES):

                    target = masks[b, c]
                    pred = predictions[b, c]

                    if target.sum() > 0:
                        positive_pairs += 1

                    else:
                        empty_pairs += 1

                        if pred.sum() == 0:
                            correct_empty_pairs += 1

                        false_positive_pixels += int(pred.sum().item())

                    total_pixels += target.numel()

    dice_per_class = (
        2 * intersection + 1.0
    ) / (
        pred_sum + target_sum + 1.0
    )

    # Foreground Dice: calculate only over classes with
    # positive validation targets.
    positive_classes = target_sum > 0

    if positive_classes.any():
        foreground_dice = dice_per_class[
            positive_classes
        ].mean().item()
    else:
        foreground_dice = 0.0

    false_positive_score = (
        1.0 - false_positive_pixels / max(total_pixels, 1)
    )

    empty_accuracy = (
        correct_empty_pairs / max(empty_pairs, 1)
    )

    official_score = (
        0.70 * foreground_dice
        + 0.20 * false_positive_score
        + 0.10 * empty_accuracy
    )

    return official_score


# -------------------- 10. Training and validation --------------------

def run_training_epoch():

    model.train()
    total_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled
        ):

            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(train_loader.dataset)


def run_validation_epoch():

    model.eval()
    total_loss = 0.0

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled
            ):

                logits = model(images)
                loss = criterion(logits, masks)

            total_loss += loss.item() * images.size(0)

    return total_loss / len(val_loader.dataset)


# -------------------- 11. W&B setup --------------------

try:
    import wandb

    wandb_run = wandb.init(
        project="23f3004085-t32026",
        name="M4-Q8-ResNet34-UNet-6epochs",
        config={
            "architecture": "ResNet34 U-Net",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "optimizer": "AdamW",
            "learning_rate": BASE_LR,
            "threshold": 0.5,
            "seed": SEED,
            "amp": amp_enabled,
            "input_size": "128x800"
        }
    )

except Exception as e:
    wandb = None
    wandb_run = None
    print("W&B logging unavailable:", e)


# -------------------- 12. Run six epochs --------------------

history = []
global_step = 0

best_score = -1.0
best_epoch = 0
best_checkpoint = "m4_q8_best_model.pth"

for epoch in range(1, EPOCHS + 1):

    # Update learning rate at every optimizer step
    for param_group in optimizer.param_groups:
        param_group["lr"] = get_lr(global_step)

    train_loss = run_training_epoch()

    global_step += steps_per_epoch

    val_loss = run_validation_epoch()

    official_score = calculate_official_score(
        model, val_loader
    )

    generalization_gap = val_loss - train_loss

    result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "official_score": official_score,
        "generalization_gap": generalization_gap
    }

    history.append(result)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Official Score: {official_score:.4f} | "
        f"Gap: {generalization_gap:.4f}"
    )

    if wandb_run is not None:
        wandb.log(result)

    # Save checkpoint with highest official score
    if official_score > best_score:

        best_score = official_score
        best_epoch = epoch

        torch.save(
            model.state_dict(),
            best_checkpoint
        )


# -------------------- 13. Results and Question 8 answers --------------------

results_df = pd.DataFrame(history)

print("\nComplete training history:")
display(results_df)

lowest_loss_row = results_df.loc[
    results_df["val_loss"].idxmin()
]

highest_score_row = results_df.loc[
    results_df["official_score"].idxmax()
]

epoch6 = results_df.iloc[-1]

print("\nQUESTION 8 RESULTS")
print("------------------")

print(
    "Lowest validation loss:",
    f"Epoch {int(lowest_loss_row['epoch'])}, "
    f"{lowest_loss_row['val_loss']:.4f}"
)

print(
    "Highest official validation score:",
    f"Epoch {int(highest_score_row['epoch'])}, "
    f"{highest_score_row['official_score']:.4f}"
)

print(
    "Epoch 6 generalization gap:",
    f"{epoch6['generalization_gap']:.4f}"
)

print(
    "Best checkpoint:",
    best_checkpoint,
    "| Epoch:",
    best_epoch
)


# -------------------- 14. Learning curves --------------------

plt.figure(figsize=(12, 5))

plt.plot(
    results_df["epoch"],
    results_df["train_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    results_df["epoch"],
    results_df["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("M4 Q8 - Training and Validation Loss")
plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    results_df["epoch"],
    results_df["official_score"],
    marker="o",
    label="Official Validation Score"
)

plt.xlabel("Epoch")
plt.ylabel("Official Score")
plt.title("M4 Q8 - Official Validation Score")
plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(True)
plt.show()


# Restore best checkpoint
model.load_state_dict(
    torch.load(best_checkpoint, map_location=device)
)

print("\nRestored best checkpoint from epoch:", best_epoch)

if wandb_run is not None:
    wandb_run.finish()

Device: cpu
Training images: 960
Validation images: 240
Train batches: 60
Validation batches: 15
W&B logging unavailable: No module named 'wandb'
Epoch 1/6 | Train Loss: 0.7853 | Val Loss: 0.7781 | Official Score: 0.1687 | Gap: -0.0072
Epoch 2/6 | Train Loss: 0.6562 | Val Loss: 0.6023 | Official Score: 0.2567 | Gap: -0.0539
Epoch 3/6 | Train Loss: 0.5774 | Val Loss: 0.5608 | Official Score: 0.3112 | Gap: -0.0165


In [ ]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Select the same 1,200-image subset
# --------------------------------------------------

image_ids_q9 = sorted(df["image_id"].unique())[:1200]

q9_df = df[df["image_id"].isin(image_ids_q9)].copy()

HEIGHT = 128
WIDTH = 800
TOTAL_PIXELS = HEIGHT * WIDTH

# --------------------------------------------------
# 2. Decode RLE and calculate defect area
# --------------------------------------------------

def get_rle_area(rle):

    if pd.isna(rle) or str(rle).strip() == "":
        return 0

    values = np.asarray(str(rle).split(), dtype=np.int64)

    starts = values[0::2] - 1
    lengths = values[1::2]

    total_area = 0

    for start, length in zip(starts, lengths):

        if start >= 0 and start < TOTAL_PIXELS:
            end = min(start + length, TOTAL_PIXELS)
            total_area += max(0, end - start)

    return total_area


# --------------------------------------------------
# 3. Calculate area for every positive image-class pair
# --------------------------------------------------

positive_pairs = q9_df[
    q9_df["mask_rle"].notna()
    & (q9_df["mask_rle"].astype(str).str.strip() != "")
].copy()

positive_pairs["defect_area"] = positive_pairs["mask_rle"].apply(
    get_rle_area
)

# --------------------------------------------------
# 4. Assign defect-size bins
# --------------------------------------------------

def assign_bin(area):

    if 1 <= area <= 100:
        return "1-100"

    elif 101 <= area <= 500:
        return "101-500"

    else:
        return "501+"


positive_pairs["size_bin"] = positive_pairs["defect_area"].apply(
    assign_bin
)

# --------------------------------------------------
# 5. Part (a): Number of positive pairs in each bin
# --------------------------------------------------

bin_order = ["1-100", "101-500", "501+"]

bin_counts = (
    positive_pairs["size_bin"]
    .value_counts()
    .reindex(bin_order, fill_value=0)
)

print("PART (a): Positive pairs in each bin")

for bin_name in bin_order:
    print(f"{bin_name}: {bin_counts[bin_name]}")


# --------------------------------------------------
# 6. Part (b): Percentage of positive pixels in 501+ bin
# --------------------------------------------------

total_positive_pixels = positive_pairs["defect_area"].sum()

large_bin_pixels = positive_pairs.loc[
    positive_pairs["size_bin"] == "501+",
    "defect_area"
].sum()

large_bin_percentage = (
    large_bin_pixels / total_positive_pixels * 100
    if total_positive_pixels > 0 else 0
)

print("\nPART (b): Percentage of positive pixels in 501+ bin")
print(f"{large_bin_percentage:.2f}%")


# --------------------------------------------------
# 7. Part (c): Expected missed positive pairs
# --------------------------------------------------

recall_by_bin = {
    "1-100": 0.30,
    "101-500": 0.70,
    "501+": 0.95
}

missed_by_bin = {}

for bin_name in bin_order:

    missed_by_bin[bin_name] = (
        bin_counts[bin_name] * (1 - recall_by_bin[bin_name])
    )

expected_missed_pairs = sum(missed_by_bin.values())

most_missed_bin = max(
    missed_by_bin,
    key=missed_by_bin.get
)

print("\nPART (c): Expected missed positive pairs")

for bin_name in bin_order:
    print(
        f"{bin_name}: "
        f"{missed_by_bin[bin_name]:.2f}"
    )

print(f"Expected total missed pairs: {expected_missed_pairs:.2f}")
print(f"Bin contributing most missed pairs: {most_missed_bin}")


# --------------------------------------------------
# 8. Part (d): Explanation
# --------------------------------------------------

print("\nPART (d):")
print(
    "The bin with the most missed pairs can differ from the bin "
    "with the most missed pixels because pair counts measure "
    "the number of missed defects, while pixel counts also depend "
    "on the size of each defect."
)


# --------------------------------------------------
# 9. Final answer in the required format
# --------------------------------------------------

print("\nFINAL ANSWER:")

print(
    f"{bin_counts['1-100']}, "
    f"{bin_counts['101-500']}, "
    f"{bin_counts['501'] if '501' in bin_counts else bin_counts['501+']}; "
    f"{large_bin_percentage:.2f}%; "
    f"{expected_missed_pairs:.2f}, {most_missed_bin}; "
    "The bin with the most missed pairs can differ from the bin "
    "with the most missed pixels because pair counts measure "
    "the number of missed defects, while pixel counts also depend "
    "on the size of each defect."
)

In [ ]:
# ============================================================
# MILESTONE 4 - QUESTION 10
# CONTROLLED EXPERIMENTATION (OFAT)
#
# R0  : Baseline - constant LR, natural sampling, AMP
# R0b : Same as R0, seed 7
# R1  : R0 + warmup/cosine scheduler
# R2  : R0 + bounded square-root sampler
# ============================================================

import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, WeightedRandomSampler

# -------------------- Configuration --------------------

EPOCHS = 6
BATCH_SIZE = 16
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-2
HEIGHT, WIDTH = 128, 800
NUM_CLASSES = 4

CLASS_NAMES = ["defect_a", "defect_b", "defect_c", "defect_d"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# AMP is enabled on CUDA. CPU fallback is FP32.
AMP_ENABLED = device.type == "cuda"

print("Device:", device)
print("AMP enabled:", AMP_ENABLED)
print("Training images:", len(m4_train_ids))
print("Validation images:", len(m4_val_ids))


# -------------------- Compound loss from M3 --------------------

class CompoundLoss(nn.Module):

    def __init__(self, alpha=0.5, beta=0.5, smooth=1.0):
        super().__init__()

        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):

        bce_loss = self.bce(logits, targets)

        probabilities = torch.sigmoid(logits)

        intersection = (probabilities * targets).sum(
            dim=(2, 3)
        )

        denominator = (
            probabilities.sum(dim=(2, 3))
            + targets.sum(dim=(2, 3))
        )

        dice = (
            2 * intersection + self.smooth
        ) / (
            denominator + self.smooth
        )

        dice_loss = 1 - dice.mean()

        return (
            self.alpha * bce_loss
            + self.beta * dice_loss
        )


# -------------------- DataLoaders --------------------

# Natural sampling for R0, R0b and R1
natural_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=AMP_ENABLED
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


# -------------------- R2: Bounded square-root sampler --------------------

# Calculate group counts using only the 960 training images.
train_combinations = m4_combinations[
    m4_combinations["image_id"].isin(m4_train_ids)
].copy()

group_counts = train_combinations[
    "stratify_group"
].value_counts()

clean_count = int(group_counts.get("clean", 1))

combination_map = train_combinations.set_index(
    "image_id"
)["stratify_group"].to_dict()

r2_weights = []

for image_id in m4_train_ids:

    group = combination_map[image_id]
    n_group = int(group_counts[group])

    weight = min(
        math.sqrt(clean_count / n_group),
        5.0
    )

    r2_weights.append(weight)

r2_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(r2_weights),
    num_samples=len(r2_weights),
    replacement=True
)

sampler_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=r2_sampler,
    num_workers=0,
    pin_memory=AMP_ENABLED
)

print("R2 sampler created.")
print("R2 weight range:", min(r2_weights), "to", max(r2_weights))


# -------------------- Official validation score --------------------

def evaluate_model(model, loader):

    model.eval()

    intersection = torch.zeros(NUM_CLASSES, device=device)
    pred_sum = torch.zeros(NUM_CLASSES, device=device)
    target_sum = torch.zeros(NUM_CLASSES, device=device)

    correct_empty_pairs = 0
    empty_pairs = 0
    false_positive_pixels = 0
    total_pixels = 0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):
                logits = model(images)

            predictions = (
                torch.sigmoid(logits.float()) >= 0.5
            ).float()

            intersection += (predictions * masks).sum(
                dim=(0, 2, 3)
            )

            pred_sum += predictions.sum(dim=(0, 2, 3))
            target_sum += masks.sum(dim=(0, 2, 3))

            # Calculate clean-image false-alarm rate
            for b in range(masks.size(0)):
                for c in range(NUM_CLASSES):

                    target = masks[b, c]
                    pred = predictions[b, c]

                    if target.sum().item() == 0:

                        empty_pairs += 1

                        if pred.sum().item() == 0:
                            correct_empty_pairs += 1

                        else:
                            false_positive_pixels += int(
                                pred.sum().item()
                            )

                    total_pixels += target.numel()

    dice_per_class = (
        2 * intersection + 1.0
    ) / (
        pred_sum + target_sum + 1.0
    )

    positive_classes = target_sum > 0

    if positive_classes.any():
        foreground_dice = dice_per_class[
            positive_classes
        ].mean().item()
    else:
        foreground_dice = 0.0

    false_positive_score = (
        1.0 - false_positive_pixels / max(total_pixels, 1)
    )

    empty_accuracy = (
        correct_empty_pairs / max(empty_pairs, 1)
    )

    official_score = (
        0.70 * foreground_dice
        + 0.20 * false_positive_score
        + 0.10 * empty_accuracy
    )

    false_alarm_rate = (
        (empty_pairs - correct_empty_pairs)
        / max(empty_pairs, 1)
    ) * 100

    return official_score, false_alarm_rate


# -------------------- Training functions --------------------

def create_model(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=1,
        classes=NUM_CLASSES
    )

    return model.to(device)


def get_scheduler_lr(step, total_steps):

    warmup_steps = 60

    if step < warmup_steps:
        multiplier = (step + 1) / warmup_steps

    else:

        progress = (
            (step - warmup_steps)
            / max(1, total_steps - warmup_steps)
        )

        progress = min(progress, 1.0)

        multiplier = 0.5 * (
            1 + math.cos(math.pi * progress)
        )

    return BASE_LR * multiplier


def train_one_experiment(
    run_name,
    seed,
    train_loader,
    use_scheduler=False
):

    print("\n" + "=" * 65)
    print("Starting:", run_name)
    print("=" * 65)

    # Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = create_model(seed)

    criterion = CompoundLoss().to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=AMP_ENABLED
    )

    total_steps = EPOCHS * len(train_loader)
    global_step = 0

    best_score = -1.0
    best_epoch = 0
    best_checkpoint = f"{run_name}_best.pth"

    epoch_history = []

    # W&B tracking
    run = None

    try:
        import wandb

        run = wandb.init(
            project="23f3004085-t32026",
            name=run_name,
            reinit=True,
            config={
                "seed": seed,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": BASE_LR,
                "optimizer": "AdamW",
                "scheduler": (
                    "warmup_cosine" if use_scheduler
                    else "constant"
                ),
                "sampling": (
                    "sqrt_bounded" if run_name == "R2"
                    else "natural"
                ),
                "amp": AMP_ENABLED,
                "threshold": 0.5
            }
        )

    except Exception as e:
        print("W&B initialization unavailable:", e)

    # Six epochs
    for epoch in range(1, EPOCHS + 1):

        model.train()

        total_train_loss = 0.0

        for images, masks in train_loader:

            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad(set_to_none=True)

            if use_scheduler:

                current_lr = get_scheduler_lr(
                    global_step,
                    total_steps
                )

                for param_group in optimizer.param_groups:
                    param_group["lr"] = current_lr

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                logits = model(images)
                loss = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += (
                loss.item() * images.size(0)
            )

            global_step += 1

        train_loss = (
            total_train_loss / len(train_loader.dataset)
        )

        # Validation loss
        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():

            for images, masks in val_loader:

                images = images.to(device)
                masks = masks.to(device)

                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.float16,
                    enabled=AMP_ENABLED
                ):

                    logits = model(images)
                    loss = criterion(logits, masks)

                total_val_loss += (
                    loss.item() * images.size(0)
                )

        val_loss = (
            total_val_loss / len(val_loader.dataset)
        )

        official_score, false_alarm_rate = evaluate_model(
            model,
            val_loader
        )

        epoch_result = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "official_score": official_score,
            "false_alarm_rate": false_alarm_rate
        }

        epoch_history.append(epoch_result)

        print(
            f"{run_name} | Epoch {epoch}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Official Score: {official_score:.4f} | "
            f"False Alarm: {false_alarm_rate:.2f}%"
        )

        if run is not None:
            wandb.log(epoch_result)

        # Save best official-score checkpoint
        if official_score > best_score:

            best_score = official_score
            best_epoch = epoch

            torch.save(
                model.state_dict(),
                best_checkpoint
            )

    # Restore best checkpoint
    model.load_state_dict(
        torch.load(best_checkpoint, map_location=device)
    )

    best_score, best_false_alarm = evaluate_model(
        model,
        val_loader
    )

    print(
        f"\n{run_name} best epoch: {best_epoch}"
    )

    print(
        f"{run_name} best official score: {best_score:.4f}"
    )

    print(
        f"{run_name} best false-alarm rate: "
        f"{best_false_alarm:.2f}%"
    )

    if run is not None:
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_official_score"] = best_score
        wandb.summary["best_false_alarm_rate"] = best_false_alarm
        run.finish()

    return {
        "name": run_name,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_score": best_score,
        "false_alarm_rate": best_false_alarm,
        "history": epoch_history,
        "checkpoint": best_checkpoint
    }


# -------------------- Run all four experiments --------------------

experiment_results = {}

# R0: Baseline
experiment_results["R0"] = train_one_experiment(
    run_name="R0",
    seed=42,
    train_loader=natural_train_loader,
    use_scheduler=False
)

# R0b: Seed-noise estimate
experiment_results["R0b"] = train_one_experiment(
    run_name="R0b",
    seed=7,
    train_loader=natural_train_loader,
    use_scheduler=False
)

# R1: Baseline + warmup/cosine
experiment_results["R1"] = train_one_experiment(
    run_name="R1",
    seed=42,
    train_loader=natural_train_loader,
    use_scheduler=True
)

# R2: Baseline + bounded square-root sampler
experiment_results["R2"] = train_one_experiment(
    run_name="R2",
    seed=42,
    train_loader=sampler_train_loader,
    use_scheduler=False
)


# -------------------- Results table --------------------

summary = pd.DataFrame([
    {
        "Run": name,
        "Best Epoch": result["best_epoch"],
        "Best Official Score": result["best_score"],
        "False Alarm Rate (%)": result["false_alarm_rate"]
    }
    for name, result in experiment_results.items()
])

print("\nFINAL EXPERIMENT COMPARISON")
display(summary)


# -------------------- Noise band and isolated gains --------------------

scores = {
    name: result["best_score"]
    for name, result in experiment_results.items()
}

noise_band = abs(scores["R0"] - scores["R0b"])

gain_r1 = scores["R1"] - scores["R0"]
gain_r2 = scores["R2"] - scores["R0"]

print("\nNOISE AND IMPROVEMENTS")
print("Noise band:", f"{noise_band:.3f}")
print("R1 gain over R0:", f"{gain_r1:.3f}")
print("R2 gain over R0:", f"{gain_r2:.3f}")

print(
    "R1 exceeds 2x noise:",
    "Yes" if gain_r1 > 2 * noise_band else "No"
)

print(
    "R2 exceeds 2x noise:",
    "Yes" if gain_r2 > 2 * noise_band else "No"
)


# -------------------- Final answer values --------------------

print("\nQUESTION 10 FINAL ANSWER VALUES")

print(
    "a:",
    f"{scores['R0']:.3f}, "
    f"{scores['R0b']:.3f}, "
    f"{scores['R1']:.3f}, "
    f"{scores['R2']:.3f}"
)

print(
    "b:",
    f"{noise_band:.3f}, "
    f"{gain_r1:.3f}, "
    f"{gain_r2:.3f}"
)

print(
    "c:",
    "Yes" if gain_r1 > 2 * noise_band else "No",
    "Yes" if gain_r2 > 2 * noise_band else "No"
)

print(
    "d:",
    f"{experiment_results['R0']['false_alarm_rate']:.1f}%, "
    f"{experiment_results['R2']['false_alarm_rate']:.1f}%"
)

print(
    "e: Changing the sampler, learning-rate schedule and AMP "
    "simultaneously confounds their individual effects, so the "
    "improvement cannot be attributed to any one component."
)

print(
    "f: A change in false-alarm rate after sampling indicates "
    "that changing the training distribution can shift the "
    "model's predictions on clean validation pairs."
)